# Block 4 — Wavelength, weather, and the scan
### How does wavelength shape what you see, and how do you assemble a volume?

The payoff block, and the strongest local hook: most radar networks across Africa run C-band (some X-band), and a shorter wavelength *attenuates*. We see the shadow, the polarimetric fix, and finally how many beams stack into one volume scan — the problem phased arrays exist to solve. Sliders only; run setup first.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

C = 2.998e8; A_E = 6371.0
ATTEN = {"S": (0.0001, 1.0), "C": (0.006, 1.0), "X": (0.03, 1.0)}  # k = a*R^b, dB/km one-way

def rain_scene(x_km, peak=80, bg=2, centre=40, width=8):
    return bg + peak*np.exp(-((x_km - centre)/width)**2)
def z_from_rain(R):
    return 10*np.log10(200 * np.maximum(R, 1e-6)**1.6)
def path_attenuation(x_km, R, band):
    a, b = ATTEN[band]; return 2*np.cumsum(a * R**b * np.gradient(x_km))
def beam_height(r_km, elev_deg):
    Re = (4/3)*A_E; phi = np.radians(elev_deg)
    return np.sqrt(r_km**2 + Re**2 + 2*r_km*Re*np.sin(phi)) - Re

VCP = {"VCP 21 (9 tilts)":  [0.5,1.45,2.4,3.35,4.3,6.0,9.9,14.6,19.5],
       "VCP 12 (14 tilts)": [0.5,0.9,1.3,1.8,2.4,3.1,4.0,5.1,6.4,8.0,10.0,12.5,15.6,19.5],
       "VCP 31 clear-air (5)": [0.5,1.5,2.5,3.5,4.5],
       "fast low-level (3)": [0.5,1.5,2.5]}

print("Block 4 ready | attenuation across cell (dB):",
      {b: round(float(path_attenuation(np.linspace(0,120,600), rain_scene(np.linspace(0,120,600)), b)[-1]),1) for b in "SCX"})

Block 4 ready | attenuation across cell (dB): {'S': 0.3, 'C': 16.5, 'X': 82.5}


## 1 — Shorter wavelength, bigger price: attenuation

Rain doesn't just reflect energy, it absorbs and scatters it *out* of the beam, so everything behind a cell is seen through a curtain of rain. That loss is negligible at S-band, real at C-band, and can wipe the echo out entirely at X-band.

In [2]:
def plot_attenuation(peak_mmh=80):
    x = np.linspace(0, 120, 600); R = rain_scene(x, peak_mmh); z_true = z_from_rain(R)
    plt.figure(figsize=(8, 4.6))
    plt.plot(x, z_true, "k--", label="true reflectivity")
    for band, col in [("S", "tab:blue"), ("C", "tab:orange"), ("X", "tab:red")]:
        plt.plot(x, z_true - path_attenuation(x, R, band), color=col, lw=2, label=band + "-band observed")
    plt.ylim(-10, 60); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range along beam (km)"); plt.ylabel("reflectivity (dBZ)")
    plt.title("attenuation carves a shadow behind the cell")
    plt.show()

interact(plot_attenuation,
         peak_mmh=FloatSlider(value=80, min=10, max=150, step=5, description="peak mm/h"));

interactive(children=(FloatSlider(value=80.0, description='peak mm/h', max=150.0, min=10.0, step=5.0), Output(…

**Basic**

1. Which band reads the cell peak closest to the truth? Which loses everything behind the core?
2. At 80 mm/h, roughly how many dBZ does the C-band echo lose behind the cell?

**A little further**

1. Most networks across Africa are C-band. What would that shadow do to a rainfall estimate further downrange of a heavy core?
2. Raise the peak rain rate. Does the X-band shadow deepen steadily, or accelerate? What does that imply for heavy convection?

## 2 — The fix: polarimetry (KDP)

Reflectivity is a *power* measurement, so attenuation eats it. KDP comes from the *phase* of the signal, which keeps advancing no matter how much power is lost — it's immune to attenuation and actually measures how much occurred, so you can add it back.

In [3]:
def plot_polarimetry(band="C", peak_mmh=80):
    x = np.linspace(0, 120, 600); R = rain_scene(x, peak_mmh)
    z_true = z_from_rain(R); pia = path_attenuation(x, R, band)
    plt.figure(figsize=(8, 4.6))
    plt.plot(x, z_true, "k--", label="true")
    plt.plot(x, z_true - pia, color="tab:red", lw=2, label="raw " + band + "-band (attenuated)")
    plt.plot(x, z_true - pia + pia, color="tab:green", lw=2, ls="-.", label="KDP-corrected")
    plt.ylim(-10, 60); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range along beam (km)"); plt.ylabel("reflectivity (dBZ)")
    plt.title("KDP / PhiDP measures the attenuation, so you can add it back")
    plt.show()

interact(plot_polarimetry,
         band=Dropdown(options=["S", "C", "X"], value="C", description="band"),
         peak_mmh=FloatSlider(value=80, min=10, max=150, step=5, description="peak mm/h"));

interactive(children=(Dropdown(description='band', index=1, options=('S', 'C', 'X'), value='C'), FloatSlider(v…

**Basic**

1. For C-band, does the KDP-corrected curve recover the truth behind the cell?
2. Switch to S-band. Is there much to correct? Why or why not?

**A little further**

1. Switch to X-band with a strong cell. The model corrects perfectly — but in reality, what limits the correction once the echo is fully extinguished?
2. Why is KDP immune to attenuation when Z is not? So where does dual-pol buy the most — S, C, or the bands African networks run?

## 3 — Putting it together: scan strategies

A volume coverage pattern is a stack of tilts. The fan (on Block 1's height axes) exposes the three things a VCP trades: the **cone of silence** above the top tilt, the **low-level gap** below the bottom one, and the **update time** that grows with every tilt — the exact problem electronically-steered phased arrays exist to solve.

In [4]:
def plot_vcp(pattern="VCP 21 (9 tilts)", rpm=3.0):
    tilts = VCP[pattern]; top, bottom = max(tilts), min(tilts); r = np.linspace(1, 300, 300)
    plt.figure(figsize=(8, 5.2))
    for e in tilts: plt.plot(r, beam_height(r, e), lw=1, color="tab:blue")
    plt.fill_between(r, beam_height(r, top), 20, color="tab:red", alpha=0.06)
    plt.text(35, 16.5, "cone of silence", color="tab:red", fontsize=9)
    plt.fill_between(r, 0, beam_height(r, bottom), color="gray", alpha=0.10)
    plt.text(150, 0.4, "low-level gap", color="dimgray", fontsize=9)
    vol = len(tilts)*60/rpm
    plt.xlim(0, 300); plt.ylim(0, 20); plt.grid(alpha=0.3)
    plt.xlabel("range (km)"); plt.ylabel("height (km)")
    plt.title(pattern + ": volume time " + str(round(vol)) + " s (" + str(round(vol/60,1))
              + " min)  |  cone half-angle " + str(round(90-top)) + "°")
    plt.show()

interact(plot_vcp,
         pattern=Dropdown(options=list(VCP), value="VCP 21 (9 tilts)", description="pattern"),
         rpm=FloatSlider(value=3, min=1, max=6, step=0.5, description="rpm"));

interactive(children=(Dropdown(description='pattern', options=('VCP 21 (9 tilts)', 'VCP 12 (14 tilts)', 'VCP 3…

**Basic**

1. Read the volume update time for VCP 21 and VCP 12 at 3 rpm. Which is faster, and why?
2. Find the cone of silence: at 30 km range, how high can a storm top be before it sits above the top tilt and is missed?

**A little further**

1. Choose a pattern that updates in under 3 minutes at 3 rpm. How many tilts can you afford, and what vertical coverage do you sacrifice?
2. Connect to Block 3: cleaner velocity needs more pulses per radial, so suppose each tilt takes 25 s instead of 20. Recompute the VCP 12 volume time — and name the technology this trade motivates.

## Facilitator notes
- This block is the synthesis: wavelength effects you can't avoid, the polarimetric tool that rescues them, and how every earlier idea stacks into one volume scan.
- The one idea to leave with: *a shorter wavelength buys a finer beam from a smaller dish but pays in attenuation — and assembling a volume is a constant trade between vertical detail, clean velocity, and update speed.*
- Capstone link: the scan-time trade is precisely why phased-array radar is the field's next direction.